# Graficos exploratorios

Este notebook concentra todo el graficado bajo demanda: mapas, series de caja (Nino 3.4 / Nino 1+2 / cualquier ventana), resumen de QC y comparacion contra ERSSTv5. Editar libremente las celdas segun se necesite.

Requiere que `data/processed/masked/` ya tenga archivos (correr `./run.sh` hasta el paso 05 como minimo).

In [15]:
from pathlib import Path

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pandas as pd

# Asume que el notebook se abre desde la carpeta donde vive este archivo
BASE_DIR = Path.cwd()
MASKED_DIR = BASE_DIR / "data/processed/masked"
QC_CSV = BASE_DIR / "data/processed/qc_report.csv"

NON_DATA_NAMES = {"lat", "lon", "x", "y", "time", "lat_bnds",
                  "lon_bnds", "time_bnds", "bnds", "area"}

# Cajas de referencia (formato de longitud 0-360, igual que config/domains.yaml)
NINO34 = dict(lon1=190, lon2=240, lat1=-5, lat2=5)
NINO12 = dict(lon1=270, lon2=280, lat1=-10, lat2=0)


def list_available_models() -> list[str]:
    return sorted(
        f.stem.replace("tos_", "", 1).replace("_historical", "")
        for f in MASKED_DIR.glob("tos_*_historical.nc")
    )


list_available_models()

['ACCESS-CM2',
 'ACCESS-ESM1-5',
 'AWI-CM-1-1-MR',
 'BCC-CSM2-MR',
 'CAMS-CSM1-0',
 'CAS-ESM2-0',
 'CESM2-WACCM',
 'CMCC-CM2-SR5',
 'CMCC-ESM2',
 'CNRM-CM6-1',
 'CNRM-CM6-1-HR',
 'CNRM-ESM2-1',
 'CanESM5',
 'CanESM5-CanOE',
 'FGOALS-f3-L',
 'FGOALS-g3',
 'FIO-ESM-2-0',
 'GFDL-CM4',
 'GFDL-ESM4',
 'GISS-E2-1-G',
 'GISS-E2-1-H',
 'GISS-E2-2-G',
 'HadGEM3-GC31-LL',
 'IITM-ESM',
 'INM-CM4-8',
 'INM-CM5-0',
 'IPSL-CM6A-LR',
 'KIOST-ESM',
 'MCM-UA-1-0',
 'MIROC-ES2L',
 'MIROC6',
 'MPI-ESM1-2-HR',
 'MPI-ESM1-2-LR',
 'MRI-ESM2-0',
 'NESM3',
 'NorESM2-LM',
 'NorESM2-MM',
 'UKESM1-0-LL']

## GRAFICO 1: Mapas

Promedio temporal del campo completo, para un modelo/experimento o para ERSSTv5.

In [3]:
from glob import glob as gb
import xarray as xr

In [16]:
def masked_path(model: str, exp: str = "historical") -> Path:
    if model.upper().startswith("ERSST"):
        return MASKED_DIR / "ersstv5_region.nc"
    return MASKED_DIR / f"tos_{model}_{exp}.nc"

def plot_map(model: str, exp: str = "historical", ax=None):
    with nc.Dataset(masked_path(model, exp)) as ds:
        varname = next(v for v in ds.variables if v not in NON_DATA_NAMES)
        # if not varname in ['tos']: 
        #     print(varname,'variable incorrecta en:', model)            
        #     varname='tos' 
        # if 'ERSST' in model: varname='sst'
        d=xr.open_dataset(masked_path(model, exp))
        lat = ds.variables["lat"][:]
        lon = ds.variables["lon"][:]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(9, 4))
    d[varname].sel(time=slice('1950-12','1950-12')).plot(ax=ax,extend='both',levels=np.arange(20,34,1),cmap='turbo')    
    ax.set_xlabel("longitud")
    ax.set_ylabel("latitud")
    ax.set_title(f"{model} {exp.upper()}")
    plt.savefig('figures/sst_t0_%s.png'%model,dpi=100, bbox_inches="tight")
    if standalone: plt.close()
# Ejemplo: cambiar el modelo/experimento segun se necesite
for mode in list_available_models():
    #print(mode)
    plot_map(mode, "historical")   
plot_map("ERSSTv5")

## GRAFICO 2: Series de caja (Nino 3.4 / Nino 1+2 / caja libre)

El promedio de caja se calcula aqui mismo, en el momento -- no hace falta que el pipeline lo deje precalculado en disco.

In [19]:
def box_mean(model: str, exp: str, lon1: float, lon2: float, lat1: float, lat2: float):
    """Devuelve (anios_decimales, valores) promediados en la caja dada."""
    with nc.Dataset(masked_path(model, exp)) as ds:        
        if not os.path.exists(masked_path(model, exp)): return
        varname = next(v for v in ds.variables if v not in NON_DATA_NAMES)
        lat = ds.variables["lat"][:]
        lon = ds.variables["lon"][:]
        data = np.ma.masked_invalid(ds.variables[varname][:])
        time = ds.variables["time"]
        dates = nc.num2date(time[:], time.units, getattr(time, "calendar", "standard"))

    lat_idx = np.where((lat >= lat1) & (lat <= lat2))[0]
    lon_idx = np.where((lon >= lon1) & (lon <= lon2))[0]
    sub = data[:, lat_idx, :][:, :, lon_idx]
    serie = sub.mean(axis=(1, 2))
    years = np.array([d.year + (d.month - 0.5) / 12 for d in dates])
    return years, serie

def plot_box_series(model: str, box: dict, box_name: str, ax=None):
    """Serie de caja del modelo con sus tres periodos superpuestos:
    historico (gris), ssp245 (azul, alpha .8), ssp585 (rojo, alpha .8)."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(9, 3))

    estilos = {
        "historical": dict(color="gray", alpha=1.0),
        "ssp245": dict(color="tab:blue", alpha=0.8),
        "ssp585": dict(color="tab:red", alpha=0.8),
    }
    for exp, estilo in estilos.items():
        years, values = box_mean(model, exp, **box)
        ax.plot(years, values, label=exp, **estilo)

    ax.set_xlabel("anio")
    ax.set_ylabel("SST (degC)")
    ax.set_title(f"{box_name} -- {model}")
    ax.legend()
    plt.savefig('figures/sst_%s_serie_%s.png' % (model, box_name.replace(" ", "")), dpi=100)
    if standalone: plt.close()

# Ejemplo
for mode in list_available_models():
    if not 'ssp' in mode: continue
    plot_box_series(mode, NINO12, "Nino 1+2")
    plot_box_series(mode, NINO34, "Nino 3.4")

## GRAFICO 3: Resumen de control de calidad

Lee `data/processed/qc_report.csv` (paso 06) y grafica PASS/FAIL por modelo.

In [7]:
def plot_qc_summary():
    df = pd.read_csv(QC_CSV)
    counts = df.groupby(["model", "status"]).size().unstack(fill_value=0)
    for col in ("PASS", "FAIL", "ERROR_CDO"):
        if col not in counts.columns:
            counts[col] = 0

    fig, ax = plt.subplots(figsize=(max(6, len(counts) * 0.6), 4))
    ax.bar(counts.index, counts["PASS"], label="PASS", color="tab:green")
    ax.bar(counts.index, counts["FAIL"] + counts["ERROR_CDO"], bottom=counts["PASS"],
           label="FAIL", color="tab:red")
    ax.set_ylabel("num. de archivos (experimentos)")
    ax.set_title("Control de calidad por modelo")
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.savefig('figures/sst_QD.png',dpi=100, bbox_inches="tight")
    plt.close()
    return df
plot_qc_summary()

,file,model,experiment,min_sst,max_sst,n_months,ok_range,ok_length,status,detail
0,tos_ACCESS-CM2_historical.nc,ACCESS-CM2,historical,15.2998,33.6541,1980,True,True,PASS,NaN
1,tos_ACCESS-CM2_ssp245.nc,ACCESS-CM2,ssp245,17.8976,35.5890,1032,True,True,PASS,NaN
2,tos_ACCESS-CM2_ssp585.nc,ACCESS-CM2,ssp585,17.3293,37.7158,1032,True,True,PASS,NaN
3,tos_ACCESS-ESM1-5_historical.nc,ACCESS-ESM1-5,historical,15.7610,33.0173,1980,True,True,PASS,NaN
4,tos_ACCESS-ESM1-5_ssp245.nc,ACCESS-ESM1-5,ssp245,18.3292,34.9659,1032,True,True,PASS,NaN
...,...,...,...,...,...,...,...,...,...,...
105,tos_NorESM2-MM_ssp245.nc,NorESM2-MM,ssp245,15.4665,34.1510,1032,True,True,PASS,NaN
106,tos_NorESM2-MM_ssp585.nc,NorESM2-MM,ssp585,15.8502,35.4096,1032,True,True,PASS,NaN
107,tos_UKESM1-0-LL_historical.nc,UKESM1-0-LL,historical,15.1333,33.0149,1980,True,True,PASS,NaN
108,tos_UKESM1-0-LL_ssp245.nc,UKESM1-0-LL,ssp245,16.4525,35.2881,1032,True,True,PASS,NaN


## GRAFICO 4: Comparacion Nino 3.4 -- periodo historico, modelos vs ERSSTv5

Solo el periodo historico: los escenarios futuros no tienen contraparte observada.

In [8]:
def plot_box_boxplot(box: dict, box_name: str):
    """Boxplot periodo historico, para la caja dada (Nino 3.4, Nino 1+2 o
    cualquier otra): una caja por fuente (modelos + ERSSTv5), restringidas
    al mismo conjunto de anios (la interseccion entre todas), para que la
    comparacion de dispersion (std) y mediana (p50) sea directa entre
    fuentes."""
    sources = ["ERSSTv5"] + list_available_models()

    series = {}
    for src in sources:
        exp = "" if src == "ERSSTv5" else "historical"
        years, values = box_mean(src, exp, **box)
        series[src] = (years, values)

    # anios (enteros) presentes en TODAS las fuentes
    common_years = None
    for src in sources:
        yrs = set(np.round(series[src][0]).astype(int))
        common_years = yrs if common_years is None else (common_years & yrs)
    common_years = sorted(common_years)

    data_for_box, labels = [], []
    for src in sources:
        years, values = series[src]
        yr_int = np.round(years).astype(int)
        mask = np.isin(yr_int, common_years)
        data_for_box.append(np.ma.compressed(values[mask]))
        labels.append(src)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot(data_for_box, tick_labels=labels)

    for i, vals in enumerate(data_for_box):
        std = np.std(vals)
        p50 = np.median(vals)
        ax.text(i + 1, p50, f"p50={p50:.2f}\nσ={std:.2f}", ha="center", va="center",
                fontsize=8, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    ax.set_ylabel(f"SST {box_name} (degC)")
    ax.set_title(f"{box_name} -- periodo historico ({len(common_years)} anios comunes: "
                 f"{common_years[0]}-{common_years[-1]})")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.savefig('figures/sst_%s_boxplot.png' % box_name.replace(" ", ""), dpi=100, bbox_inches="tight")
    plt.close()
plot_box_boxplot(NINO34, "Nino 3.4")
plot_box_boxplot(NINO12, "Nino 1+2")

IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed